# Synthetic Hierarchical Data Generation

Generates tables for testing **Skill 6 — Hierarchical Reconciliation** with the new multi-level design.

Produces a 4-level hierarchy (Total → Country → Region → Store) with independently generated
time series per level, additively coherent by construction.

**Output tables (one pair per level + membership table):**
- `{use_case}_store_best_models` + `{use_case}_store_evaluation_output`
- `{use_case}_region_best_models` + `{use_case}_region_evaluation_output`
- `{use_case}_country_best_models` + `{use_case}_country_evaluation_output`
- `{use_case}_total_best_models` + `{use_case}_total_evaluation_output`
- `{use_case}_membership` — adjacency list (unique_id, level_name, parent_unique_id)

## Parameters

In [ ]:
def _get(key, default):
    try:
        return dbutils.widgets.get(key)
    except Exception:
        return default

catalog             = _get('catalog', 'catalog_timeseries')
schema              = _get('schema', 'synthetic')
use_case            = _get('use_case', 'mmf_hierarchical')
n_countries         = int(_get('n_countries', '2'))
n_regions_per_country = int(_get('n_regions_per_country', '2'))
n_stores_per_region = int(_get('n_stores_per_region', '3'))
n_months            = int(_get('n_months', '36'))
prediction_length   = int(_get('prediction_length', '3'))

print(f'catalog={catalog}  schema={schema}  use_case={use_case}')
print(f'hierarchy: 1 total / {n_countries} countries / {n_countries * n_regions_per_country} regions / '
      f'{n_countries * n_regions_per_country * n_stores_per_region} stores')
print(f'n_months={n_months}  prediction_length={prediction_length}')

## Generate Time Series (all levels)

In [ ]:
import numpy as np
import pandas as pd
from datetime import datetime
from dateutil.relativedelta import relativedelta

np.random.seed(42)

# Build hierarchy structure
countries = [f'Country_{chr(65+i)}' for i in range(n_countries)]
regions   = [f'Region_{c}_{j+1}' for c in countries for j in range(n_regions_per_country)]
stores    = [f'Store_{r}_{k+1}' for r in regions for k in range(n_stores_per_region)]

# Date range: historical + future
start_date = datetime(2021, 1, 1)
all_dates  = [start_date + relativedelta(months=m) for m in range(n_months + prediction_length)]
hist_dates = all_dates[:n_months]
future_dates = all_dates[n_months:]

# Generate store-level actuals (trend + seasonality + noise)
store_actuals = {}
for store in stores:
    base  = np.random.uniform(50, 200)
    slope = np.random.uniform(0.2, 1.0)
    amp   = np.random.uniform(5, 20)
    noise = np.random.uniform(2, 8)
    t = np.arange(n_months)
    y = base + slope * t + amp * np.sin(2 * np.pi * t / 12) + np.random.normal(0, noise, n_months)
    store_actuals[store] = np.maximum(y, 0)

# Explicit parent mappings (avoids ambiguity from string splitting on compound names)
region_of  = {f'Store_{r}_{k+1}': r for r in regions  for k in range(n_stores_per_region)}
country_of = {f'Region_{c}_{j+1}': c for c in countries for j in range(n_regions_per_country)}

# Aggregate upward (additively coherent by construction)
region_actuals = {}
for region in regions:
    region_stores = [s for s in stores if region_of[s] == region]
    region_actuals[region] = sum(store_actuals[s] for s in region_stores)

country_actuals = {}
for country in countries:
    country_regions = [r for r in regions if country_of[r] == country]
    country_actuals[country] = sum(region_actuals[r] for r in country_regions)

total_actuals = {'Total': sum(country_actuals[c] for c in countries)}

print(f'Generated: {len(stores)} stores, {len(regions)} regions, {len(countries)} countries, 1 total')
print(f'History: {n_months} months | Future: {prediction_length} months')

## Write Level Tables

In [ ]:
from pyspark.sql import functions as F
from pyspark.sql.types import StructType, StructField, StringType, DoubleType, TimestampType, ArrayType

spark.sql(f'CREATE SCHEMA IF NOT EXISTS {catalog}.{schema}')

MODEL_NAME = 'StatsForecastAutoArima'
N_BACKTEST_WINDOWS = 4

def write_level(level_name, actuals_dict):
    series_ids = list(actuals_dict.keys())

    # Each level is produced by its own run_forecast call, so it has its own run_id.
    # best_models and evaluation_output for a level share the same run_id (they are
    # co-produced in one run) — this is what enables the (unique_id, model, run_id) join.
    level_run_id = f'{use_case}_{level_name}_run'

    # --- best_models: one row per series; ds/y are arrays (matches real MMF output) ---
    bm_rows = []
    for uid in series_ids:
        actuals = actuals_dict[uid]
        last_val = actuals[-1]
        ds_list = list(future_dates)
        y_list = [max(0.0, float(last_val + np.random.normal(0, last_val * 0.03))) for _ in future_dates]
        bm_rows.append({
            'unique_id': uid,
            'model': MODEL_NAME,
            'avg_metric': float(np.random.uniform(0.05, 0.15)),
            'ds': ds_list,
            'y': y_list,
            'forecast_source': 'main_pipeline',
            'run_id': level_run_id,
        })
    bm_pdf = pd.DataFrame(bm_rows)
    bm_table = f'{catalog}.{schema}.{use_case}_{level_name}_best_models'
    (
        spark.createDataFrame(bm_pdf)
        .withColumn('ds', F.col('ds').cast(ArrayType(TimestampType())))
        .withColumn('y', F.col('y').cast(ArrayType(DoubleType())))
        .write.format('delta').mode('overwrite').option('overwriteSchema', 'true').saveAsTable(bm_table)
    )
    print(f'  Written: {bm_table} ({len(bm_rows)} rows)')

    # --- evaluation_output: backtest windows with forecast/actual arrays ---
    eval_rows = []
    for uid in series_ids:
        actuals = actuals_dict[uid]
        for w in range(N_BACKTEST_WINDOWS):
            end_idx = n_months - w * prediction_length
            start_idx = end_idx - prediction_length
            if start_idx < 0:
                continue
            actual_vals = actuals[start_idx:end_idx].tolist()
            forecast_vals = [max(0.0, v + np.random.normal(0, v * 0.05)) for v in actual_vals]
            eval_rows.append({
                'unique_id': uid,
                'backtest_window_start_date': hist_dates[start_idx],
                'forecast': forecast_vals,
                'actual': actual_vals,
                'model': MODEL_NAME,
                'run_id': level_run_id,
            })
    eval_pdf = pd.DataFrame(eval_rows)
    eval_table = f'{catalog}.{schema}.{use_case}_{level_name}_evaluation_output'
    (
        spark.createDataFrame(eval_pdf)
        .withColumn('forecast', F.col('forecast').cast(ArrayType(DoubleType())))
        .withColumn('actual', F.col('actual').cast(ArrayType(DoubleType())))
        .write.format('delta').mode('overwrite').option('overwriteSchema', 'true').saveAsTable(eval_table)
    )
    print(f'  Written: {eval_table} ({len(eval_rows)} rows)')

print('Writing store level...')
write_level('store', store_actuals)
print('Writing region level...')
write_level('region', region_actuals)
print('Writing country level...')
write_level('country', country_actuals)
print('Writing total level...')
write_level('total', total_actuals)

In [ ]:
# --- Membership table (adjacency list) ---
membership_rows = []

# Root
membership_rows.append({'unique_id': 'Total', 'level_name': 'total', 'parent_unique_id': None})

# Countries -> Total
for c in countries:
    membership_rows.append({'unique_id': c, 'level_name': 'country', 'parent_unique_id': 'Total'})

# Regions -> Country
for r in regions:
    parent = country_of[r]
    membership_rows.append({'unique_id': r, 'level_name': 'region', 'parent_unique_id': parent})

# Stores -> Region
for s in stores:
    parent = region_of[s]
    membership_rows.append({'unique_id': s, 'level_name': 'store', 'parent_unique_id': parent})

membership_pdf = pd.DataFrame(membership_rows)
membership_table = f'{catalog}.{schema}.{use_case}_membership'
spark.createDataFrame(membership_pdf).write.format('delta').mode('overwrite').option('overwriteSchema', 'true').saveAsTable(membership_table)
print(f'Written: {membership_table} ({len(membership_rows)} rows)')
display(membership_pdf.head(10))

## Verify

In [ ]:
print('=== Membership table ===')
display(spark.sql(f"""
    SELECT level_name, COUNT(*) AS n_series
    FROM {catalog}.{schema}.{use_case}_membership
    GROUP BY level_name ORDER BY n_series DESC
"""))

for level in ['store', 'region', 'country', 'total']:
    bm_count  = spark.table(f'{catalog}.{schema}.{use_case}_{level}_best_models').count()
    eval_count = spark.table(f'{catalog}.{schema}.{use_case}_{level}_evaluation_output').count()
    print(f'{level}: best_models={bm_count} rows | evaluation_output={eval_count} rows')

print('\nReady for Skill 6. Use these levels config:')
print(f"""
levels = [
    {{"name": "store",   "best_models_table": "{catalog}.{schema}.{use_case}_store_best_models",   "evaluation_table": "{catalog}.{schema}.{use_case}_store_evaluation_output"}},
    {{"name": "region",  "best_models_table": "{catalog}.{schema}.{use_case}_region_best_models",  "evaluation_table": "{catalog}.{schema}.{use_case}_region_evaluation_output"}},
    {{"name": "country", "best_models_table": "{catalog}.{schema}.{use_case}_country_best_models", "evaluation_table": "{catalog}.{schema}.{use_case}_country_evaluation_output"}},
    {{"name": "total",   "best_models_table": "{catalog}.{schema}.{use_case}_total_best_models",   "evaluation_table": "{catalog}.{schema}.{use_case}_total_evaluation_output"}},
]
hierarchy_table = "{catalog}.{schema}.{use_case}_membership"
""")